# Predictive Maintenance with the AI4I 2020 Dataset

**Activity 4.3 — Developing and Testing AI Solutions**
Kevin Curtis, Xtrac Limited

---

This workbook trains and evaluates a machine failure classifier on a published
industrial dataset. It is the public benchmark half of the report: the same
pipeline is later applied to synthetic Xtrac gearbox rig data.

**The dataset.** AI4I 2020 Predictive Maintenance Dataset, from the UCI Machine
Learning Repository (Matzka, 2020). 10,000 observations from a milling machine,
14 columns. Synthetic, but modelled on real industrial data. Licensed CC BY 4.0.

**The question.** Given the machine's current operating conditions, will this
process fail?

**Run every cell in order.** Cells marked **YOUR TURN** are for you to complete
or change — that is where the learning happens, and where your report gets its
material.

### Setup

Put `ai4i2020.csv` in the same folder as this notebook. If you do not have it,
download it from the UCI repository page for the AI4I 2020 dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             average_precision_score, precision_recall_curve,
                             ConfusionMatrixDisplay)
from sklearn.base import clone

RANDOM_SEED = 42
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

print("libraries loaded")

---
## 1. Load and look at the data

Never model data you have not looked at. This section is the exploratory work
that feeds section 3.1 of your report.

In [ ]:
machines = pd.read_csv("ai4i2020.csv")

print("rows and columns:", machines.shape)
machines.head()

### What the columns mean

| Column | Meaning |
|---|---|
| `UDI` | Row number. Not a feature. |
| `Product ID` | Serial number. Not a feature. |
| `Type` | Product quality variant: L (low), M (medium), H (high). |
| `Air temperature [K]` | Ambient temperature in Kelvin. |
| `Process temperature [K]` | Machine process temperature, roughly 10 K above air. |
| `Rotational speed [rpm]` | Spindle speed. |
| `Torque [Nm]` | Applied torque. |
| `Tool wear [min]` | Cumulative minutes of tool use. |
| `Machine failure` | **The target.** 1 if the process failed. |
| `TWF, HDF, PWF, OSF, RNF` | Which failure mode occurred. |

The five failure-mode columns are the reason the process failed. They are only
known *after* the failure. Using them as features would be data leakage: the
model would score near-perfectly and be useless in practice, because at
prediction time you do not have them.

In [ ]:
print(machines.dtypes, "\n")
print("missing values:", machines.isna().sum().sum())
machines.describe().round(2)

---
## 2. How imbalanced is it?

This single number drives most of the decisions that follow.

In [ ]:
failure_counts = machines["Machine failure"].value_counts()
failure_rate = machines["Machine failure"].mean()

print(failure_counts)
print(f"\nfailure rate: {failure_rate:.3%}")
print(f"\nA model that predicts 'no failure' every time would be "
      f"{1 - failure_rate:.1%} accurate and would catch zero failures.")

**This is the argument against accuracy**, and it belongs in section 3.2 of
your report. Roughly 97 in every 100 processes succeed, so a model that never
predicts failure scores 96.6% accuracy while being completely worthless.

In [ ]:
failure_modes = machines[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum()
print(failure_modes)
print("\nsum of individual modes:", failure_modes.sum())
print("rows flagged as failed:", machines['Machine failure'].sum())
print("\nThe totals differ because some rows have more than one mode set,")
print("and a few are inconsistent. Matzka notes this as label noise.")

> **YOUR TURN 1**
>
> Failure rate differs by product quality variant. Work out the failure rate for
> each `Type` (L, M, H) and note which is worst. This matters: it means `Type`
> carries real predictive information, and it is a categorical column, so it
> will need one-hot encoding.
>
> Hint: `machines.groupby(...)[...].mean()`

In [ ]:
# YOUR TURN 1 — failure rate by product type


---
## 3. Choose the features

Two decisions here, both of which need justifying in your report.

**Dropped as leakage:** the five failure-mode flags. They encode the answer.

**Dropped as identifiers:** `UDI` and `Product ID`. A serial number carries no
information about physics, and letting a model learn from row numbers is a
classic way to get an impressive score that means nothing.

In [ ]:
target_column = "Machine failure"

leakage_columns = ["TWF", "HDF", "PWF", "OSF", "RNF"]
identifier_columns = ["UDI", "Product ID"]

numeric_features = ["Air temperature [K]", "Process temperature [K]",
                    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
categorical_features = ["Type"]

input_columns = numeric_features + categorical_features

print("features used:", len(input_columns))
for c in input_columns:
    print("  ", c)
print("\nexcluded:", leakage_columns + identifier_columns)

---
## 4. Split before you touch anything

The test rows are set aside now and not looked at again until the model is
trained. Any preparation step that learns from data — the scaler's mean, an
imputer's median — must learn from the training rows only. Otherwise
information from the test set leaks into training, the scores come out
flattering, and the model disappoints in production.

`stratify` keeps the failure rate the same in both halves. Without it, a random
cut of a 3.4% positive class can easily land lopsided.

In [ ]:
inputs = machines[input_columns]
target = machines[target_column]

training_inputs, testing_inputs, training_target, testing_target = train_test_split(
    inputs, target,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=target)

print(f"training rows: {len(training_inputs)}   testing rows: {len(testing_inputs)}")
print(f"failures  training {training_target.sum()} ({training_target.mean():.3%})"
      f"   testing {testing_target.sum()} ({testing_target.mean():.3%})")

---
## 5. The baseline

Every model has to beat this. It is not a formality — quoting it is what stops
a 96.6% accuracy figure from looking impressive.

In [ ]:
always_no_failure = np.zeros(len(testing_target))

print(f"accuracy: {(always_no_failure == testing_target).mean():.3f}")
print(f"failures caught: {int(((always_no_failure == 1) & (testing_target == 1)).sum())}"
      f" out of {int(testing_target.sum())}")
print(f"recall: {recall_score(testing_target, always_no_failure, zero_division=0):.3f}")

---
## 6. Build the pipeline

A `Pipeline` bolts the preparation steps to the algorithm so they travel as one
object. Call `.fit()` and the scaler learns its mean and standard deviation from
the training rows; call `.predict()` and it reuses those same numbers. You
cannot accidentally leak.

- **StandardScaler** rewrites each numeric column as "how many standard
  deviations from its own mean". Rotational speed sits around 1500 and torque
  around 40; without scaling, logistic regression's penalty falls unevenly on
  them purely because of their units.
- **OneHotEncoder** turns `Type` into three 0/1 columns. Numbering L=1, M=2, H=3
  would tell the model H is three times L, which is meaningless.
- **class_weight="balanced"** makes each failure count more in the loss
  function, in proportion to how rare it is. Without it the model can minimise
  its loss almost entirely by predicting "no failure".

Trees do not need scaling — a split at "torque above 60" is unaffected by
rescaling — but it does no harm, and keeping one pipeline for both algorithms
keeps the comparison clean.

In [ ]:
def make_pipeline(algorithm):
    preparation = ColumnTransformer([
        ("numbers", StandardScaler(), numeric_features),
        ("text", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_features),
    ])
    return Pipeline([("preparation", preparation),
                     ("algorithm", clone(algorithm))])

algorithms = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        C=1.0,
        random_state=RANDOM_SEED),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=-1),
}

print("pipeline defined")

---
## 7. Train and evaluate

**Precision** — of the processes flagged as failing, how many really failed.
Low precision means false alarms.

**Recall** — of the processes that really failed, how many were caught.
Low recall means misses.

**F1** — the two combined into one number, so models can be ranked.

**PR-AUC** (average precision) — performance across every possible cut-off, not
just the default 0.5. The right summary metric for a rare positive class;
ROC-AUC flatters models when negatives dominate.

In [ ]:
results = []
trained = {}

for name, algorithm in algorithms.items():
    pipeline = make_pipeline(algorithm)
    pipeline.fit(training_inputs, training_target)

    failure_probability = pipeline.predict_proba(testing_inputs)[:, 1]
    prediction = (failure_probability >= 0.5).astype(int)

    cross_validated = cross_val_score(
        pipeline, training_inputs, training_target,
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED),
        scoring="average_precision")

    results.append({
        "algorithm": name,
        "precision": precision_score(testing_target, prediction, zero_division=0),
        "recall": recall_score(testing_target, prediction),
        "f1": f1_score(testing_target, prediction),
        "pr_auc": average_precision_score(testing_target, failure_probability),
        "cv_pr_auc": cross_validated.mean(),
        "cv_std": cross_validated.std(),
    })
    trained[name] = (pipeline, failure_probability)

results_table = pd.DataFrame(results).round(3)
results_table

`cv_pr_auc` is cross-validation: the training data was split five ways and the
model trained five times. If it sits close to the held-out `pr_auc` and `cv_std`
is small, the result is not a fluke of one lucky split. Quote both in your
report.

---
## 8. Look at the errors, not just the score

A confusion matrix says *which way* the model is wrong. That matters more than
the headline number, because the two error types cost different amounts.

In [ ]:
best_name = results_table.sort_values("pr_auc", ascending=False).iloc[0]["algorithm"]
best_pipeline, best_probability = trained[best_name]
best_prediction = (best_probability >= 0.5).astype(int)

print(f"best by PR-AUC: {best_name}\n")

matrix = confusion_matrix(testing_target, best_prediction)
print(pd.DataFrame(matrix,
                   index=["actually OK", "actually FAILED"],
                   columns=["predicted OK", "predicted FAIL"]))

print("\n", classification_report(testing_target, best_prediction,
                                   target_names=["OK", "FAILED"], digits=3))

In [ ]:
figure, axis = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(matrix,
                       display_labels=["OK", "FAILED"]).plot(ax=axis, colorbar=False)
axis.set_title(f"{best_name} at threshold 0.5")
plt.tight_layout()
plt.show()

> **YOUR TURN 2**
>
> Write down, in your own words, what each of the four boxes means for a
> maintenance engineer standing at the machine. Which of the two error types
> would you rather have, and why? There is no correct answer here — it depends
> on what a stoppage costs versus what an unnecessary inspection costs. Your
> reasoning is the material for section 3.2 of the report.

---
## 9. The threshold is a decision, not a default

The model outputs a probability. Turning that into a yes/no needs a cut-off, and
0.5 is a library default that nobody chose for this problem.

Moving it trades the two error types against each other. Nothing is retrained —
the model is identical, only the line moves.

In [ ]:
sweep = []
for threshold in np.arange(0.05, 0.96, 0.05):
    prediction = (best_probability >= threshold).astype(int)
    sweep.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(testing_target, prediction, zero_division=0),
        "recall": recall_score(testing_target, prediction),
        "f1": f1_score(testing_target, prediction, zero_division=0),
        "false_alarms": int(((prediction == 1) & (testing_target == 0)).sum()),
        "missed_failures": int(((prediction == 0) & (testing_target == 1)).sum()),
    })

threshold_table = pd.DataFrame(sweep).round(3)
threshold_table

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(threshold_table.threshold, threshold_table.precision, marker="o", label="precision")
axes[0].plot(threshold_table.threshold, threshold_table.recall, marker="s", label="recall")
axes[0].plot(threshold_table.threshold, threshold_table.f1, marker="^", label="F1")
axes[0].axvline(0.5, ls="--", c="grey")
axes[0].set_xlabel("decision threshold")
axes[0].set_title("Metrics against threshold")
axes[0].legend()

axes[1].plot(threshold_table.threshold, threshold_table.false_alarms,
             marker="o", label="false alarms")
axes[1].plot(threshold_table.threshold, threshold_table.missed_failures,
             marker="s", label="missed failures")
axes[1].set_xlabel("decision threshold")
axes[1].set_ylabel(f"count (test set n={len(testing_target)})")
axes[1].set_title("Error trade-off")
axes[1].legend()

plt.tight_layout()
plt.show()

> **YOUR TURN 3**
>
> Pick a threshold and justify it. Look at the table, decide what operating
> point you would recommend to a maintenance team, and write two sentences
> saying why. This is a candidate refinement for section 5 of your report, so
> record the before-and-after numbers.

In [ ]:
# YOUR TURN 3 — set your chosen threshold and report the result
chosen_threshold = 0.5   # change this

chosen_prediction = (best_probability >= chosen_threshold).astype(int)
print(f"threshold {chosen_threshold}")
print(f"  precision {precision_score(testing_target, chosen_prediction, zero_division=0):.3f}")
print(f"  recall    {recall_score(testing_target, chosen_prediction):.3f}")
print(f"  false alarms    {int(((chosen_prediction == 1) & (testing_target == 0)).sum())}")
print(f"  missed failures {int(((chosen_prediction == 0) & (testing_target == 1)).sum())}")

---
## 10. Precision-recall curve

The grey line is what random guessing achieves, and it sits at the failure rate.
Distance above it is the model's real contribution. PR-AUC is the area under
this curve.

In [ ]:
precision_points, recall_points, _ = precision_recall_curve(
    testing_target, best_probability)

figure, axis = plt.subplots(figsize=(6, 4.5))
axis.plot(recall_points, precision_points, label=best_name)
axis.axhline(testing_target.mean(), ls="--", c="grey", label="no-skill baseline")
axis.set_xlabel("recall")
axis.set_ylabel("precision")
axis.set_title("Precision-recall curve")
axis.legend()
plt.tight_layout()
plt.show()

---
## 11. What did the model learn?

If the important features match what an engineer would expect, that supports
adoption. If they do not, either the model has found something real or it has
found a shortcut — and both are worth investigating before anyone trusts it.

In [ ]:
algorithm_object = best_pipeline.named_steps["algorithm"]
encoded_names = list(best_pipeline.named_steps["preparation"]
                     .named_transformers_["text"]
                     .get_feature_names_out(categorical_features))
feature_names = numeric_features + encoded_names

if hasattr(algorithm_object, "feature_importances_"):
    importances = pd.Series(algorithm_object.feature_importances_, index=feature_names)
    label = "importance"
else:
    importances = pd.Series(np.abs(algorithm_object.coef_[0]), index=feature_names)
    label = "coefficient size"

importances = importances.sort_values(ascending=False)
print(importances.round(4).to_string())

figure, axis = plt.subplots(figsize=(7, 4))
importances[::-1].plot(kind="barh", ax=axis)
axis.set_xlabel(label)
axis.set_title(f"What {best_name} relies on")
plt.tight_layout()
plt.show()

Note the limitation: a random forest tells you *which* features matter but not
*which direction*. Logistic regression coefficients give you direction, which is
easier to explain to a non-technical audience. Worth a sentence in section 2.3
on communicating to different audiences.

---
## 12. Feature engineering

The dataset gives raw readings. Physics suggests some combinations should carry
more signal:

- **Temperature difference** (process minus air). Poor heat dissipation shows up
  as a small difference, and HDF is one of the documented failure modes.
- **Mechanical power** = torque x angular velocity. PWF is defined by power
  falling outside a band, so power should be more informative than either
  torque or speed alone.
- **Wear x torque**. OSF is defined by this product exceeding a limit.

Note what is happening here: domain knowledge is being encoded as arithmetic.
That is the honest version of feature engineering, and it is exactly the kind of
mathematical reasoning K18 is asking for.

In [ ]:
engineered = machines.copy()

engineered["temp_difference_K"] = (engineered["Process temperature [K]"]
                                   - engineered["Air temperature [K]"])

angular_velocity = engineered["Rotational speed [rpm]"] * 2 * np.pi / 60   # rad/s
engineered["power_W"] = engineered["Torque [Nm]"] * angular_velocity

engineered["wear_times_torque"] = (engineered["Tool wear [min]"]
                                   * engineered["Torque [Nm]"])

engineered_numeric = numeric_features + ["temp_difference_K", "power_W",
                                         "wear_times_torque"]
engineered[["temp_difference_K", "power_W", "wear_times_torque"]].describe().round(2)

In [ ]:
def evaluate(numeric_columns, dataframe, algorithm, label):
    columns = numeric_columns + categorical_features
    train_x, test_x, train_y, test_y = train_test_split(
        dataframe[columns], dataframe[target_column],
        test_size=0.25, random_state=RANDOM_SEED, stratify=dataframe[target_column])

    preparation = ColumnTransformer([
        ("numbers", StandardScaler(), numeric_columns),
        ("text", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_features)])
    pipeline = Pipeline([("preparation", preparation), ("algorithm", clone(algorithm))])
    pipeline.fit(train_x, train_y)

    probability = pipeline.predict_proba(test_x)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    return {"features": label,
            "n": len(numeric_columns) + 1,
            "precision": precision_score(test_y, prediction, zero_division=0),
            "recall": recall_score(test_y, prediction),
            "f1": f1_score(test_y, prediction),
            "pr_auc": average_precision_score(test_y, probability)}


comparison = pd.DataFrame([
    evaluate(numeric_features, engineered, algorithms[best_name], "raw only"),
    evaluate(engineered_numeric, engineered, algorithms[best_name], "raw + engineered"),
]).round(3)
comparison

> **YOUR TURN 4**
>
> Did the engineered features help? Record the before and after PR-AUC either
> way.
>
> If they helped, this is your section 5 refinement, and the physics is the
> justification.
>
> If they did not, that is still a result. Say why you expected them to work and
> what the data says instead. A rejected hypothesis with an explanation is
> stronger evidence of testing than a tidy improvement.

---
## 13. Experiments to run

Change one thing, rerun, record what happened. Each of these produces material
for a specific section of your report. Keep notes as you go — what you expected,
what happened, why.

In [ ]:
# EXPERIMENT A — class weighting
# Set class_weight=None on the random forest and rerun section 7.
# Watch recall collapse. This is your class-imbalance evidence for section 4.4.

# EXPERIMENT B — tree depth
# Change min_samples_leaf from 2 to 1, then to 50.
# 1 overfits, 50 underfits. That is bias-variance you can see. Section 4.2.

# EXPERIMENT C — regularisation strength
# Change C in LogisticRegression from 1.0 to 0.001, then to 1000.
# C is the inverse of penalty strength: small C means heavy penalty,
# which shrinks coefficients toward zero. Section 4.4.

# EXPERIMENT D — leakage, deliberately
# Add the five failure-mode flags to numeric_features and rerun.
# The score will look superb. Explain in your report why it is worthless.

# EXPERIMENT E — the seed
# Change RANDOM_SEED to 1, then 7, then 2024.
# How much do the numbers move? That tells you how many decimal places
# you are entitled to quote. Section 3.5.


> **YOUR TURN 5**
>
> Run at least three of the experiments above. For each one record: what you
> changed, what you expected, what happened, and what it means. That table goes
> straight into your appendix.

---
## 14. Saving results for the report

Writes the tables and charts you need. Change the folder first.

In [ ]:
OUTPUT_FOLDER = "."   # change to your own folder

results_table.to_csv(f"{OUTPUT_FOLDER}/ai4i_results.csv", index=False)
threshold_table.to_csv(f"{OUTPUT_FOLDER}/ai4i_threshold_sweep.csv", index=False)
comparison.to_csv(f"{OUTPUT_FOLDER}/ai4i_feature_comparison.csv", index=False)
pd.DataFrame(matrix,
             index=["actually OK", "actually FAILED"],
             columns=["predicted OK", "predicted FAIL"]
             ).to_csv(f"{OUTPUT_FOLDER}/ai4i_confusion_matrix.csv")

print("written to", OUTPUT_FOLDER)

---
## Reference

Matzka, S. (2020) 'Explainable Artificial Intelligence for Predictive Maintenance
Applications', *2020 Third International Conference on Artificial Intelligence
for Industries (AI4I)*, pp. 69-74.

Dataset: AI4I 2020 Predictive Maintenance Dataset, UCI Machine Learning
Repository. Licensed CC BY 4.0.

---

### Where this fits in the report

| Notebook section | Report section |
|---|---|
| 1-3 | 3.1 test data design and source |
| 2, 5, 7 | 3.2 metrics and why accuracy fails |
| 7-8 | 3.3 results |
| 9 | 4.3 decision boundaries and thresholds |
| 6, 13 | 4.4 scaling, regularisation, class weights |
| 11 | 2.3 explaining to different audiences |
| 12 | 5 refinement strategy |
| 13 | Appendix B experiment log |